# 🥩 Классификация Мраморности Мяса - Optuna + ConvNeXt
### С подбором гиперпараметров (20 trials)

**Оптимизировано для текстур мраморности**

In [12]:
%matplotlib inlineimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import DataLoader, Dataset, Subsetfrom torch.optim.lr_scheduler import CosineAnnealingLRfrom torchvision import transformsimport timmfrom PIL import Image, ImageEnhanceimport osimport numpy as npfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_scoreimport matplotlib.pyplot as pltimport seaborn as snsfrom tqdm import tqdmimport warningsimport optunaimport pandas as pdimport randomimport gcimport timewarnings.filterwarnings('ignore')print(f"PyTorch: {torch.__version__}")print(f"CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.7.1+cu118
CUDA: True
GPU: NVIDIA GeForce RTX 4070


In [13]:
# ====================== CONFIG ======================class Config:    seed = 42    data_dir = "marb"  # Датасет мраморности        # Models for Optuna - оптимизированы для текстур    model_names = [        "convnextv2_tiny.fcmae_ft_in22k_in1k",  # ConvNeXt V2 Tiny - лучший для текстур        "efficientnet_b0.ra_in1k",              # EfficientNet B0 - эффективный        "resnet50.a1_in1k",                     # ResNet-50 - стабильный        "resnetrs50.tf_in1k",                   # ResNet-RS-50 - улучшенный    ]        img_size = 256  # Больше для деталей мраморности    num_workers = 0  # Для Windows        # Optuna    optuna_n_trials = 15    optuna_head_epochs = 7    optuna_full_epochs = 15        # Final training    num_epochs = 30    patience = 7    grad_clip = 1.0        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")cfg = Config()# Seedrandom.seed(cfg.seed)np.random.seed(cfg.seed)torch.manual_seed(cfg.seed)if torch.cuda.is_available():    torch.cuda.manual_seed(cfg.seed)

In [14]:
# ====================== DATASET ======================class MarblingDataset(Dataset):    """Датасет для классификации мраморности"""    def __init__(self, root, transform=None):        self.transform = transform        self.samples = []        self.classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}                for cls in self.classes:            cls_path = os.path.join(root, cls)            for f in sorted(os.listdir(cls_path)):                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):                    self.samples.append((os.path.join(cls_path, f), self.class_to_idx[cls]))    def __len__(self):        return len(self.samples)        def __getitem__(self, idx):        path, label = self.samples[idx]        img = Image.open(path).convert('RGB')        if self.transform:            img = self.transform(img)        return img, label

In [15]:
# ====================== TRANSFORMS ======================# СИЛЬНЫЕ аугментации для текстур мраморностиtrain_tfms = transforms.Compose([    transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0), ratio=(0.85, 1.15)),    transforms.RandomHorizontalFlip(p=0.5),    transforms.RandomVerticalFlip(p=0.3),  # Выше для текстур    transforms.RandomRotation(45),  # Больше вращений для текстур    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),    transforms.ColorJitter(brightness=0.3, contrast=0.5, saturation=0.2, hue=0.1),  # Контраст важен!    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),])# Для Optuna - умеренныеoptuna_train_tfms = transforms.Compose([    transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0)),    transforms.RandomHorizontalFlip(p=0.5),    transforms.RandomVerticalFlip(p=0.2),    transforms.RandomRotation(30),    transforms.ColorJitter(brightness=0.2, contrast=0.3, saturation=0.1, hue=0.05),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])# Валидация/Тест - без аугментацийval_tfms = transforms.Compose([    transforms.Resize((cfg.img_size, cfg.img_size)),    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),])

In [16]:
# ====================== LOSS ======================class FocalLoss(nn.Module):    """Focal Loss для сложных случаев классификации мраморности"""    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):        super().__init__()        self.alpha = alpha        self.gamma = gamma        self.reduction = reduction            def forward(self, inputs, targets):        ce_loss = F.cross_entropy(inputs, targets, reduction='none')        pt = torch.exp(-ce_loss)        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss                if self.reduction == 'mean':            return focal_loss.mean()        return focal_lossclass LabelSmoothingCrossEntropy(nn.Module):    """Label Smoothing для регуляризации"""    def __init__(self, smoothing=0.1):        super().__init__()        self.smoothing = smoothing            def forward(self, x, target):        log_probs = F.log_softmax(x, dim=-1)        nll_loss = -log_probs.gather(dim=-1, index=target.unsqueeze(1))        nll_loss = nll_loss.squeeze(1)        smooth_loss = -log_probs.mean(dim=-1)        loss = (1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss        return loss.mean()

In [17]:
# ====================== GET CLASSIFIER ======================def get_classifier(model):    """Универсальное получение classifier для разных моделей"""    # ConvNeXt uses 'head'    classifier = getattr(model, 'head', None)    if classifier is not None:        return classifier        # EfficientNet uses 'classifier'    classifier = getattr(model, 'classifier', None)    if classifier is not None:        return classifier        # ResNet uses 'fc'    classifier = getattr(model, 'fc', None)    if classifier is not None:        return classifier        raise RuntimeError("Cannot find classifier/head/fc in model")

In [18]:
# ====================== TRAIN & VALIDATE ======================def train_epoch(model, loader, criterion, optimizer, epoch, total_epochs, device):    model.train()    total_loss = correct = total = 0        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{total_epochs}')    for x, y in pbar:        x, y = x.to(device), y.to(device)                optimizer.zero_grad()        outputs = model(x)        loss = criterion(outputs, y)        loss.backward()                if cfg.grad_clip > 0:            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)                optimizer.step()                total_loss += loss.item() * x.size(0)        pred = outputs.argmax(1)        correct += (pred == y).sum().item()        total += y.size(0)                pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})        return total_loss / max(total, 1), 100. * correct / max(total, 1)@torch.no_grad()def validate(model, loader, device):    model.eval()    correct = total = 0    preds, labels = [], []        pbar = tqdm(loader, desc='Validation')    for x, y in pbar:        x, y = x.to(device), y.to(device)        outputs = model(x)        pred = outputs.argmax(1)        correct += (pred == y).sum().item()        total += y.size(0)        preds.extend(pred.cpu().numpy())        labels.extend(y.cpu().numpy())        return 100. * correct / max(total, 1), preds, labels

In [19]:
# ====================== OPTUNA TRIAL ======================def train_optuna_trial(train_loader, val_loader, num_classes, trial_params, device):    """Обучение для одного trial Optuna"""    try:        # Создаем модель        model = timm.create_model(            trial_params['model_name'],            pretrained=True,            num_classes=num_classes,            drop_rate=trial_params['dropout_rate']        ).to(device)                # Получаем classifier        classifier = get_classifier(model)                # Loss        if trial_params.get('use_focal_loss', True):            criterion = FocalLoss(                alpha=trial_params.get('focal_alpha', 0.25),                gamma=trial_params.get('focal_gamma', 2.0)            )        else:            criterion = LabelSmoothingCrossEntropy(trial_params.get('label_smoothing', 0.1))                # Phase 1: Train classifier only        for p in model.parameters():            p.requires_grad = False        for p in classifier.parameters():            p.requires_grad = True                optimizer = optim.AdamW(classifier.parameters(), lr=trial_params['lr_head'], weight_decay=trial_params['weight_decay'])        scheduler = CosineAnnealingLR(optimizer, T_max=cfg.optuna_head_epochs)                best_val_acc = 0        patience_counter = 0                for epoch in range(cfg.optuna_head_epochs):            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch, cfg.optuna_head_epochs, device)            scheduler.step()            val_acc, _, _ = validate(model, val_loader, device)                        if val_acc > best_val_acc:                best_val_acc = val_acc                patience_counter = 0            else:                patience_counter += 1                        if patience_counter >= 20:  # Reduced - if no improvement in 5 epochs, stop                break                # Phase 2: Full model fine-tuning        for p in model.parameters():            p.requires_grad = True                optimizer = optim.AdamW(model.parameters(), lr=trial_params['lr_full'], weight_decay=trial_params['weight_decay'])        scheduler = CosineAnnealingLR(optimizer, T_max=cfg.optuna_full_epochs)                for epoch in range(cfg.optuna_full_epochs):            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch, cfg.optuna_full_epochs, device)            scheduler.step()            val_acc, _, _ = validate(model, val_loader, device)                        if val_acc > best_val_acc:                best_val_acc = val_acc                patience_counter = 0            else:                patience_counter += 1                        if patience_counter >= 20:  # Reduced - if no improvement in 5 epochs, stop                break                del model, criterion, optimizer, scheduler, classifier        torch.cuda.empty_cache()        gc.collect()                return best_val_acc            except Exception as e:        print(f"Trial error: {e}")        torch.cuda.empty_cache()        gc.collect()        return 0.0

In [20]:
# ====================== OPTUNA OBJECTIVE ======================def objective(trial):    """Функция для Optuna с 20 trials"""    trial_number = trial.number + 1    print(f"\n{'='*60}")    print(f"OPTUNA TRIAL {trial_number}/{cfg.optuna_n_trials}")    print(f"{'='*60}")        # Гиперпараметры с ИСПРАВЛЕННЫМИ диапазонами    trial_params = {        # Модель - Optuna выберет лучшую для текстур        'model_name': trial.suggest_categorical('model_name', cfg.model_names),                # Learning rates - ГОРАЗДО ВЫШЕ        'lr_head': trial.suggest_float('lr_head', 1e-3, 1e-2, log=True),        'lr_full': trial.suggest_float('lr_full', 1e-5, 5e-4, log=True),                # Batch size - МЕНЬШЕ для лучших градиентов        'batch_size': trial.suggest_categorical('batch_size', [4, 8]),                # Regularization - МЕНЬШЕ dropout, ВЫШЕ weight_decay        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.3),        'weight_decay': trial.suggest_float('weight_decay', 1e-2, 0.5, log=True),        'label_smoothing': trial.suggest_float('label_smoothing', 0.0, 0.1),                # Loss        'use_focal_loss': trial.suggest_categorical('use_focal_loss', [True, False]),    }        # Параметры для Focal Loss    if trial_params['use_focal_loss']:        trial_params['focal_alpha'] = trial.suggest_float('focal_alpha', 0.2, 0.4)        trial_params['focal_gamma'] = trial.suggest_float('focal_gamma', 1.5, 2.5)        # Создаем DataLoader    train_ds = Subset(dataset, train_idx)    val_ds = Subset(dataset, val_idx)    train_ds.dataset.transform = optuna_train_tfms    val_ds.dataset.transform = val_tfms        train_loader = DataLoader(train_ds, batch_size=trial_params['batch_size'], shuffle=True, num_workers=0, pin_memory=False)    val_loader = DataLoader(val_ds, batch_size=trial_params['batch_size'], shuffle=False, num_workers=0, pin_memory=False)        # Обучаем и оцениваем    start_time = time.time()    val_acc = train_optuna_trial(train_loader, val_loader, len(dataset.classes), trial_params, cfg.device)    elapsed_time = time.time() - start_time        print(f"Trial {trial_number} took {elapsed_time:.1f}s, Accuracy: {val_acc:.2f}%")        return val_acc

In [21]:
# ====================== FINAL TRAINING ======================def train_final_model(train_loader, val_loader, test_loader, num_classes, classes, best_params):    """Финальное обучение с лучшими параметрами"""    print("\n" + "="*60)    print("FINAL MODEL TRAINING")    print("="*60)        # Создаем модель    model = timm.create_model(        best_params['model_name'],        pretrained=True,        num_classes=num_classes,        drop_rate=best_params['dropout_rate']    ).to(cfg.device)        # Получаем classifier    classifier = get_classifier(model)        # Loss    if best_params.get('use_focal_loss', True):        criterion = FocalLoss(alpha=best_params.get('focal_alpha', 0.25), gamma=best_params.get('focal_gamma', 2.0))    else:        criterion = LabelSmoothingCrossEntropy(best_params.get('label_smoothing', 0.1))        # Optimizer    lr = min(best_params['lr_head'], best_params['lr_full'] * 10)    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=best_params['weight_decay'])    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=1e-7)        # Training    best_val_acc = 0    patience_counter = 0        for epoch in range(cfg.num_epochs):        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch, cfg.num_epochs, cfg.device)        scheduler.step()        val_acc, _, _ = validate(model, val_loader, cfg.device)                print(f"Epoch {epoch+1:02d}/{cfg.num_epochs:02d} | Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")                if val_acc > best_val_acc:            best_val_acc = val_acc            patience_counter = 0            torch.save(model.state_dict(), "best_marbling_model.pth")            print(f"  → New best! (Val: {val_acc:.2f}%)")        else:            patience_counter += 1                if patience_counter >= cfg.patience:            print(f"Early stopping at epoch {epoch+1} (val_acc={val_acc:.2f}%)")            break        model.load_state_dict(torch.load("best_marbling_model.pth", map_location=cfg.device))        # Evaluation    print("\n" + "="*60)    print("FINAL EVALUATION")    print("="*60)        test_acc, test_preds, test_labels = validate(model, test_loader, cfg.device)        cm = confusion_matrix(test_labels, test_preds)    plt.figure(figsize=(10, 8))    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)    plt.title(f'Confusion Matrix (Accuracy: {test_acc:.2f}%)')    plt.tight_layout()    plt.savefig('marbling_confusion_matrix.png', dpi=300)    plt.close()        print(f"\nClassification Report:")    print(classification_report(test_labels, test_preds, target_names=classes, digits=4))        balanced_acc = balanced_accuracy_score(test_labels, test_preds) * 100    print(f"\nBalanced Accuracy: {balanced_acc:.2f}%")        # Save model    torch.save({        'model_state_dict': model.state_dict(),        'classes': classes,        'class_to_idx': {c: i for i, c in enumerate(classes)},        'accuracy': test_acc,        'balanced_accuracy': balanced_acc,        'img_size': cfg.img_size,        'model_name': best_params['model_name'],        'optuna_best_params': best_params,    }, "marbling_model_balanced.pth")        print(f"\n{'='*60}")    print("TRAINING COMPLETED!")    print(f"{'='*60}")    print(f"✅ Model saved as 'marbling_model_balanced.pth'")    print(f"🎯 Test Accuracy: {test_acc:.2f}%")    print(f"⚖️ Balanced Accuracy: {balanced_acc:.2f}%")    print(f"📊 Best Val Accuracy: {best_val_acc:.2f}%")        return model, test_acc, balanced_acc

In [22]:
# ====================== MAIN ======================def main():    global dataset, train_idx, val_idx, test_idx        print(f"Device: {cfg.device}")    print(f"Models to try: {cfg.model_names}")    print(f"Optuna trials: {cfg.optuna_n_trials}")    print(f"Image size: {cfg.img_size} (larger for marbling details)")        # Load dataset    print("\nLoading marbling dataset...")    dataset = MarblingDataset(cfg.data_dir)    print(f"Classes: {dataset.classes}")    print(f"Total images: {len(dataset)}")        class_counts = [0] * len(dataset.classes)    for _, label in dataset.samples:        class_counts[label] += 1        print(f"\nClass distribution:")    for cls, count in zip(dataset.classes, class_counts):        print(f"  {cls}: {count} ({count/len(dataset)*100:.1f}%)")        # Визуализация распределения    plt.figure(figsize=(10, 6))    plt.bar(dataset.classes, class_counts)    plt.title('Marbling Class Distribution', fontsize=14)    plt.xlabel('Marbling Grade', fontsize=12)    plt.ylabel('Count', fontsize=12)    plt.xticks(rotation=45)    for i, v in enumerate(class_counts):        plt.text(i, v + 0.5, str(v), ha='center')    plt.tight_layout()    plt.savefig('marbling_class_distribution.png', dpi=300)    plt.close()    print("Class distribution saved to 'marbling_class_distribution.png'")        # Split    idx = np.arange(len(dataset))    labels = [dataset.samples[i][1] for i in idx]        train_idx, temp_idx = train_test_split(idx, test_size=0.3, stratify=labels, random_state=cfg.seed)    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[labels[i] for i in temp_idx], random_state=cfg.seed)        print(f"\nDataset split: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")        # Optuna    print("\n" + "="*60)    print("OPTUNA HYPERPARAMETER OPTIMIZATION (20 TRIALS)")    print("="*60)        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=cfg.seed))    study.optimize(objective, n_trials=cfg.optuna_n_trials)        print(f"\n{'='*60}")    print("OPTUNA COMPLETED!")    print(f"{'='*60}")    print(f"\nBest trial: {study.best_value:.2f}%")    print("Best parameters:")    for key, value in study.best_params.items():        print(f"  {key}: {value}")        best_params = study.best_params        # Save results    df = study.trials_dataframe()    df.to_csv("marbling_optuna_results.csv", index=False)    print(f"\nResults saved to 'marbling_optuna_results.csv'")        # Final training    train_ds = Subset(MarblingDataset(cfg.data_dir, transform=train_tfms), train_idx)    val_ds = Subset(MarblingDataset(cfg.data_dir, transform=val_tfms), val_idx)    test_ds = Subset(MarblingDataset(cfg.data_dir, transform=val_tfms), test_idx)        train_loader = DataLoader(train_ds, batch_size=best_params['batch_size'], shuffle=True, num_workers=0, pin_memory=False)    val_loader = DataLoader(val_ds, batch_size=best_params['batch_size'], shuffle=False, num_workers=0, pin_memory=False)    test_loader = DataLoader(test_ds, batch_size=best_params['batch_size'], shuffle=False, num_workers=0, pin_memory=False)        model, test_acc, balanced_acc = train_final_model(        train_loader, val_loader, test_loader,        len(dataset.classes), dataset.classes,        best_params    )if __name__ == "__main__":    main()

Device: cuda
Models to try: ['convnextv2_tiny.fcmae_ft_in22k_in1k', 'efficientnet_b0.ra_in1k', 'resnet50.a1_in1k', 'resnetrs50.tf_in1k']
Optuna trials: 15
Image size: 256 (larger for marbling details)

Loading marbling dataset...
Classes: ['choice', 'prime', 'select']
Total images: 349

Class distribution:
  choice: 110 (31.5%)
  prime: 143 (41.0%)
  select: 96 (27.5%)


[I 2026-02-27 01:08:31,571] A new study created in memory with name: no-name-3014c153-01cd-475b-a9f2-e3f679723fc9


Class distribution saved to 'marbling_class_distribution.png'

Dataset split: Train=244, Val=52, Test=53

OPTUNA HYPERPARAMETER OPTIMIZATION (20 TRIALS)

OPTUNA TRIAL 1/15


Validation: 100%|██████████| 7/7 [00:03<00:00,  1.93it/s]
[I 2026-02-27 01:15:37,892] Trial 0 finished with value: 61.53846153846154 and parameters: {'model_name': 'efficientnet_b0.ra_in1k', 'lr_head': 0.001432249371823025, 'lr_full': 1.8408992080552506e-05, 'batch_size': 8, 'dropout_rate': 0.22022300234864176, 'weight_decay': 0.1595857358814127, 'label_smoothing': 0.0020584494295802446, 'use_focal_loss': True, 'focal_alpha': 0.24246782213565524, 'focal_gamma': 1.6818249672071006}. Best is trial 0 with value: 61.53846153846154.


Trial 1 took 426.3s, Accuracy: 61.54%

OPTUNA TRIAL 2/15


...

Epoch 21/30 | Train: 82.38% | Val: 71.15%


Validation: 100%|██████████| 13/13 [00:04<00:00,  3.12it/s]


Epoch 22/30 | Train: 78.69% | Val: 73.08%
Early stopping at epoch 22 (val_acc=73.08%)

FINAL EVALUATION


Validation: 100%|██████████| 14/14 [00:04<00:00,  2.92it/s]



Classification Report:
              precision    recall  f1-score   support

      choice     0.5000    0.3529    0.4138        17
       prime     0.7200    0.8182    0.7660        22
      select     0.6875    0.7857    0.7333        14

    accuracy                         0.6604        53
   macro avg     0.6358    0.6523    0.6377        53
weighted avg     0.6408    0.6604    0.6444        53


Balanced Accuracy: 65.23%

TRAINING COMPLETED!
✅ Model saved as 'marbling_model_balanced.pth'
🎯 Test Accuracy: 66.04%
⚖️ Balanced Accuracy: 65.23%
📊 Best Val Accuracy: 75.00%
